# importações de bibliotecas e tratamento das bases

In [96]:
import pandas as pd

df = pd.read_csv("Base_de_Transacoes_e_Cupons_Capturados.csv", sep=";")
df_simulacao = pd.read_csv("Base_Simulada_-_Pedestres_Av__Paulista.csv", sep=";")

df = df[["celular", "data", "hora"]].reset_index()

df_simulacao = df_simulacao[df_simulacao["possui_app_picmoney"] == "Sim"]
df_simulacao = df_simulacao[["celular", "data", "horario"]].rename(columns={"horario": "hora"}).reset_index(drop=True)


df = pd.concat([
    df_simulacao, df
])

df["data"] =pd.to_datetime(df["data"], format="%d/%m/%Y")

# Visão Geral de Desempenho

In [97]:
totais_transacoes = len(df)
usuarios_totais_unicos = df["celular"].nunique()
total_dias = df["data"].nunique()

print("Transações Totais:", totais_transacoes)
print("Total de Usuários Únicos:", usuarios_totais_unicos)
print("Total de dias calculados:", total_dias)


Transações Totais: 159957
Total de Usuários Únicos: 64745
Total de dias calculados: 31


# Usuários Ativos Diários (DAU)

In [98]:

dau_por_dia = df.groupby("data")["celular"].nunique().reset_index()

dau_por_dia.columns = ["data", "usuarios_ativos"]

DAU = dau_por_dia["usuarios_ativos"].mean()


print("DAU =", round(DAU, 2))

DAU = 4091.39


# Usuários Ativos Semanais (WAU)

In [99]:
df["semana"] = df["data"].dt.isocalendar().week

wau_por_semana = df.groupby("semana")["celular"].nunique().reset_index()

wau_por_semana.columns = ["semana", "usuarios_ativos"]

WAU = wau_por_semana["usuarios_ativos"].mean()


print("WAU = ", round(WAU, 2))

WAU =  16371.8


# Usuários Ativos Mensais (MAU)

In [100]:
df["mes"] = df["data"].dt.to_period("M")

mau_por_mes = df.groupby("mes")["celular"].nunique().reset_index()

mau_por_mes.columns = ["mes", "usuarios_ativos"]


MAU = mau_por_mes["usuarios_ativos"].mean()

print("MAU =", round(MAU, 2))
mau_por_mes

MAU = 64745.0


,mes,usuarios_ativos
0,2025-07,64745


# Taxa de Retenção de Usuários

In [101]:
df["semana"] = df["data"].dt.isocalendar().week

usuarios_iniciais = df[df["semana"] == df["semana"].min()]["celular"].unique()

usuarios_finais = df[df["semana"] == df["semana"].max()]["celular"].unique()

usuarios_retornaram = len(set(usuarios_iniciais) & set(usuarios_finais))

taxa_retencao = (usuarios_retornaram / len(usuarios_iniciais)) * 100

print(f"Usuários Iniciais: {len(usuarios_iniciais)}")
print(f"Usuários que Retornaram: {usuarios_retornaram}")
print(f"Taxa de Retenção: {taxa_retencao:.2f}%")

Usuários Iniciais: 4416
Usuários que Retornaram: 3753
Taxa de Retenção: 84.99%


# Taxa de Parada de Utilização (Churn)

In [102]:
ultima_data_usuario = df.groupby("celular")["data"].max().reset_index()
ultima_data_usuario.columns = ["celular", "ultima_data"]

data_fim_periodo = pd.to_datetime("31/07/2025", format="%d/%m/%Y")

ultima_data_usuario["dias_inativos"] = (data_fim_periodo - ultima_data_usuario["ultima_data"]).dt.days

usuarios_churned = ultima_data_usuario[ultima_data_usuario["dias_inativos"] >= 14]

taxa_churn = (len(usuarios_churned) / len(ultima_data_usuario)) * 100

print(f"Usuários Totais: {len(ultima_data_usuario)}")
print(f"Usuários Inativos (≥14 dias): {len(usuarios_churned)}")
print(f"Taxa de Churn: {taxa_churn:.2f}%")

Usuários Totais: 64745
Usuários Inativos (≥14 dias): 61
Taxa de Churn: 0.09%


# Tempo Médio na Aplicação

In [103]:
ultima_data_usuario = df.groupby("celular")["data"].max().reset_index()
ultima_data_usuario.columns = ["celular", "ultima_data"]

data_fim_periodo = pd.to_datetime("31/07/2025", format="%d/%m/%Y")

ultima_data_usuario["dias_inativos"] = (data_fim_periodo - ultima_data_usuario["ultima_data"]).dt.days

usuarios_churned = ultima_data_usuario[ultima_data_usuario["dias_inativos"] >= 14]

taxa_churn = (len(usuarios_churned) / len(ultima_data_usuario)) * 100

print(f"Usuários Totais: {len(ultima_data_usuario)}")
print(f"Usuários Inativos (≥14 dias): {len(usuarios_churned)}")
print(f"Taxa de Churn: {taxa_churn:.2f}%")

Usuários Totais: 64745
Usuários Inativos (≥14 dias): 61
Taxa de Churn: 0.09%


# Tempo Médio na Aplicação

In [104]:

df["data"] = pd.to_datetime(df["data"], dayfirst=True, errors="coerce")
df["hora_td"] = pd.to_timedelta(df["hora"].astype(str), errors="coerce")

df["timestamp"] = df["data"].dt.normalize() + df["hora_td"]

df = df.sort_values(["celular", "timestamp"]).copy()
df["gap_horas"] = df.groupby("celular")["timestamp"].diff().dt.total_seconds() / 3600

df["nova_sessao"] = df["gap_horas"].isna() | (df["gap_horas"] > 4)
df["session_id"] = df.groupby("celular")["nova_sessao"].cumsum()

sessoes = (
    df.groupby(["celular", "session_id"])["timestamp"]
      .agg(inicio="min", fim="max")
      .reset_index()
)
sessoes["duracao_min"] = (sessoes["fim"] - sessoes["inicio"]).dt.total_seconds() / 60

tempo_medio_min = round(sessoes["duracao_min"].mean(), 2)
horas = int(tempo_medio_min // 60)
mins  = int(round(tempo_medio_min % 60, 0))

print(f"Tempo médio por sessão = {tempo_medio_min} min (~{horas}h {mins}min)")


Tempo médio por sessão = 14.15 min (~0h 14min)


# Sessão por Usuário

In [105]:
df["data"] = pd.to_datetime(df["data"], dayfirst=True, errors="coerce")
df["hora"] = pd.to_timedelta(df["hora"].astype(str), errors="coerce")
df["timestamp"] = df["data"].dt.normalize() + df["hora"]


df = df.sort_values(["celular", "timestamp"])
df["gap_horas"] = df.groupby("celular")["timestamp"].diff().dt.total_seconds() / 3600

df["nova_sessao"] = df["gap_horas"].isna() | (df["gap_horas"] > 4)

df["session_id"] = df.groupby("celular")["nova_sessao"].cumsum()

sessoes_por_usuario = df.groupby("celular")["session_id"].nunique().reset_index()
sessoes_por_usuario.columns = ["celular", "qtd_sessoes"]

media_sessoes_usuario = round(sessoes_por_usuario["qtd_sessoes"].mean(), 2)

print(f"Total de Sessões: {df['session_id'].nunique()}")
print(f"Total de Usuários Únicos: {df['celular'].nunique()}")
print(f"Sessões por Usuário (média): {media_sessoes_usuario}")


Total de Sessões: 54
Total de Usuários Únicos: 64745
Sessões por Usuário (média): 2.19


# Resumo Executivo dos Índices

In [106]:
fmt_f2  = lambda x: f"{float(x):.2f}".replace(".", ",")
fmt_pct = lambda x: f"{float(x):.2f}%".replace(".", ",")

# ----- tabela no formato do modelo -----
resumo_executivo = pd.DataFrame([
    {
        "Índice":  "Usuários ativos diários (DAU)",
        "Valor":   f"{fmt_f2(DAU)} usuários",
    },
    {
        "Índice":  "Usuários ativos semanais (WAU)",
        "Valor":   f"{fmt_f2(WAU)} usuários",
    },
    {
        "Índice":  "Usuários ativos mensais (MAU)",
        "Valor":   f"{fmt_f2(MAU)} usuários",
    },
    {
        "Índice":  "Taxa de retenção (30 dias)",
        "Valor":   fmt_pct(taxa_retencao),
    },
    {
        "Índice":  "Taxa de churn (≥14 dias)",
        "Valor":   fmt_pct(taxa_churn),
    },
    {
        "Índice":  "Tempo médio na aplicação",
        "Valor":   f"{fmt_f2(tempo_medio_min)} min",
    },
    {
        "Índice":  "Sessões por usuário",
        "Valor":   f"{fmt_f2(media_sessoes_usuario)} sessôes/usuário",
    },
])

resumo_executivo


,Índice,Valor
0,Usuários ativos diários (DAU),"4091,39 usuários"
1,Usuários ativos semanais (WAU),"16371,80 usuários"
2,Usuários ativos mensais (MAU),"64745,00 usuários"
3,Taxa de retenção (30 dias),"84,99%"
4,Taxa de churn (≥14 dias),"0,09%"
5,Tempo médio na aplicação,"14,15 min"
6,Sessões por usuário,"2,19 sessôes/usuário"
